In [ ]:
# hide
import numpy as np
import pyquist as pq

In [ ]:
score, metronome = pq.Score.from_midi("../assets/twinkle-band.mid")
print(f"{len(score)} notes, {metronome.tick_to_seconds(score.end_time):.1f} seconds")
for time, kwargs in score[:4]:
    print(time, kwargs)

# Which instruments does this file use? (MIDI numbers programs from 0)
GM_NAMES = {0: "Acoustic Grand Piano", 32: "Acoustic Bass"}
for channel, program, is_drum in sorted({(kw["channel"], kw["program"], kw["is_drum"])
                                         for _, kw in score}):
    name = "Drum Kit" if is_drum else GM_NAMES.get(program, "?")
    print(f"channel {channel + 1}: program {program} ({name})")

In [ ]:
# collapse
def piano(pitch: int, duration: float, velocity: int) -> pq.Audio:
    t = np.arange(int((duration + 0.5) * 44100)) / 44100
    f = pq.helper.pitch_to_frequency(pitch)
    x = sum(np.sin(2 * np.pi * k * f * t) / k ** 2 for k in range(1, 8)) * np.exp(-3 * t)
    return pq.Audio(0.2 * (velocity / 127) * x, sample_rate=44100)


def bass(pitch: int, duration: float, velocity: int) -> pq.Audio:
    t = np.arange(int(duration * 44100)) / 44100
    f = pq.helper.pitch_to_frequency(pitch)
    x = (np.sin(2 * np.pi * f * t) + 0.3 * np.sin(4 * np.pi * f * t)) * np.exp(-2 * t)
    return pq.Audio(0.3 * (velocity / 127) * x, sample_rate=44100)


def drum(pitch: int, velocity: int) -> pq.Audio:
    t = np.arange(int(0.3 * 44100)) / 44100
    if pitch == 36:  # kick: a sine whose pitch drops quickly
        x = np.sin(2 * np.pi * (50 * t + 60 * (1 - np.exp(-30 * t)) / 30)) * np.exp(-12 * t)
    else:            # snare and hi-hat: bursts of noise
        x = np.random.uniform(-1, 1, len(t)) * np.exp(-(40 if pitch == 42 else 15) * t)
    return pq.Audio(0.3 * (velocity / 127) * x, sample_rate=44100)

In [ ]:
def instrument(pitch: int, duration: float, velocity: int, program: int, is_drum: bool,
               **kwargs) -> pq.Audio:
    """Dispatch each note to a synthesizer, based on its MIDI program."""
    if is_drum:
        return drum(pitch, velocity)
    if 32 <= program <= 39:  # programs 32 to 39 are the General MIDI basses
        return bass(pitch, duration, velocity)
    return piano(pitch, duration, velocity)


pq.play(score.render(instrument, metronome=metronome), normalize=True)

In [ ]:
# all_events=True also returns the messages that aren't notes
events, metronome = pq.Score.from_midi("../assets/twinkle-band.mid", all_events=True)
print(sorted({kwargs["mtype"] for _, kwargs in events}))

for time, kwargs in events:
    if kwargs["mtype"] == "control_change" and kwargs["control"] == 64:
        state = "down" if kwargs["value"] >= 64 else "up"
        print(f"{metronome.tick_to_seconds(time):.2f} s: sustain pedal {state} "
              f"(channel {kwargs['channel'] + 1})")